# EXEMPLO 2: Estratégias de Chunking Comparadas
## Aula 5 — HybridChunker vs. RecursiveCharacter vs. Artigo/Parágrafo
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração:** 20 minutos | **Tipo:** Referência Comparativa  
Compara as três principais estratégias de chunking para documentos jurídicos:
Docling HybridChunker, LangChain RecursiveCharacterTextSplitter e chunker por Artigo.

In [ ]:
!pip install docling>=2.0.0 langchain langchain-text-splitters reportlab --quiet
print("✅ Pronto")

In [ ]:
import warnings, re, json
warnings.filterwarnings('ignore')

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Texto jurídico de exemplo — trecho da Lei de Drogas
TEXTO_LEI = '''LEI Nº 11.343, DE 23 DE AGOSTO DE 2006
Institui o Sistema Nacional de Políticas Públicas sobre Drogas.

CAPÍTULO III - DOS CRIMES E DAS PENAS

Art. 33. Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender,
expor à venda, oferecer, ter em depósito, transportar, trazer consigo, guardar,
prescrever, ministrar, entregar a consumo ou fornecer drogas, ainda que gratuitamente,
sem autorização ou em desacordo com determinação legal ou regulamentar:
Pena - reclusão de 5 (cinco) a 15 (quinze) anos e pagamento de 500 (quinhentos) a
1.500 (mil e quinhentos) dias-multa.

§ 1º Nas mesmas penas incorre quem:
I - importa, exporta, remete, produz, fabrica, adquire, vende, expõe à venda,
oferece, fornece, tem em depósito, transporta, traz consigo ou guarda, ainda que
gratuitamente, sem autorização ou em desacordo com determinação legal ou regulamentar,
matéria-prima, insumo ou produto químico destinado à preparação de drogas;

§ 4º Nos delitos definidos no caput e no § 1º deste artigo, as penas poderão ser
reduzidas de um sexto a dois terços, vedada a conversão em penas restritivas de direitos,
desde que o agente seja primário, de bons antecedentes, não se dedique às atividades
criminosas nem integre organização criminosa.

Art. 35. Associarem-se duas ou mais pessoas para o fim de praticar, reiteradamente
ou não, qualquer dos crimes previstos nos arts. 33, caput e § 1º, e 34 desta Lei:
Pena - reclusão, de 3 (três) a 10 (dez) anos, e pagamento de 700 (setecentos) a
1.200 (mil e duzentos) dias-multa.
'''

print(f"Texto de exemplo: {len(TEXTO_LEI)} chars")

## 1. Estratégia A: LangChain RecursiveCharacterTextSplitter

A abordagem mais simples: divide por tamanho fixo, tentando preservar parágrafos.

In [ ]:
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks_recursive = splitter_recursive.split_text(TEXTO_LEI)

print(f"RecursiveCharacterTextSplitter: {len(chunks_recursive)} chunks")
for i, chunk in enumerate(chunks_recursive):
    print(f"\n[{i+1}] ({len(chunk)} chars): {chunk[:150]}...")

## 2. Estratégia B: Chunker por Artigo (jurídico)

Divide especificamente nos separadores legais brasileiros (Art., §, Inciso).

In [ ]:
SEPARADORES_JURIDICOS = [
    r'\n(?=Art\.\s+\d+[\xba\xb0]?)',
    r'\n(?=\xc2\xa7\s*\d+[\xba\xb0]?)',  # §
    r'\n(?=Parágrafo único\.)',
    r'\n(?=CAP[ÍI]TULO)',
    r'\n\n',
    r'\n',
]

splitter_juridico = RecursiveCharacterTextSplitter(
    separators=SEPARADORES_JURIDICOS,
    chunk_size=400,
    chunk_overlap=50,
    is_separator_regex=True
)
chunks_juridico = splitter_juridico.split_text(TEXTO_LEI)

print(f"ChunkerJuridico: {len(chunks_juridico)} chunks")
for i, chunk in enumerate(chunks_juridico):
    print(f"\n[{i+1}] ({len(chunk)} chars): {chunk[:200]}...")

## 3. Estratégia C: Docling HybridChunker

O chunker nativo do Docling aproveita a estrutura semântica detectada no documento.

In [ ]:
# Para usar o HybridChunker precisamos de um DoclingDocument
# Vamos criar um PDF com o texto da lei e converter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.enums import TA_JUSTIFY

CAMINHO_LEI_PDF = '/tmp/lei_drogas_exemplo.pdf'
doc_rl = SimpleDocTemplate(CAMINHO_LEI_PDF, pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
s = getSampleStyleSheet()
st_corpo = ParagraphStyle('c', parent=s['Normal'], fontSize=9,
    alignment=TA_JUSTIFY, leading=13, spaceAfter=4)

elementos = []
for paragrafo in TEXTO_LEI.strip().split('\n\n'):
    if paragrafo.strip():
        if paragrafo.startswith('LEI') or paragrafo.startswith('CAPÍTULO'):
            elementos.append(Paragraph(paragrafo.strip(), s['Heading2']))
        else:
            elementos.append(Paragraph(paragrafo.strip(), st_corpo))
        elementos.append(Spacer(1, 0.2*cm))

doc_rl.build(elementos)

# Converter com Docling
converter = DocumentConverter()
resultado = converter.convert(CAMINHO_LEI_PDF)
doc_docling = resultado.document

# HybridChunker
chunker = HybridChunker(tokenizer='BAAI/bge-m3', max_tokens=300, merge_peers=True)
chunks_hybrid = list(chunker.chunk(doc_docling))

print(f"HybridChunker: {len(chunks_hybrid)} chunks")
for i, chunk in enumerate(chunks_hybrid):
    print(f"\n[{i+1}] ({len(chunk.text)} chars): {chunk.text[:200]}...")

## 4. Comparação Final das 3 Estratégias

Qual estratégia melhor preserva a integridade jurídica dos artigos?

In [ ]:
print("=" * 70)
print("COMPARAÇÃO DE ESTRATÉGIAS DE CHUNKING")
print("=" * 70)

comparacao = [
    ('RecursiveCharacter', chunks_recursive),
    ('ChunkerJuridico', chunks_juridico),
    ('HybridChunker (Docling)', [c.text for c in chunks_hybrid]),
]

print(f"\n{'Estratégia':<30} {'Chunks':<8} {'Média chars':<14} {'Min':<8} {'Max'}")
print("-" * 75)
for nome, chunks in comparacao:
    tamanhos = [len(c) for c in chunks]
    media = sum(tamanhos)/len(tamanhos) if tamanhos else 0
    print(f"{nome:<30} {len(chunks):<8} {media:<14.0f} {min(tamanhos):<8} {max(tamanhos)}")

print("\n💡 Observações:")
print("   • RecursiveCharacter: pode cortar no meio de artigos")
print("   • ChunkerJuridico: preserva artigos, mas requer tuning de separadores")
print("   • HybridChunker: melhor para documentos com estrutura rica (relatórios, laudos)")
print("\n📌 Recomendação: use HybridChunker para a maioria dos documentos jurídicos,")
print("   e ChunkerJuridico especificamente para grandes volumes de legislação.")